# Embed Corpus & Upload to Qdrant

Pipeline:
1. Đọc corpus từ CSV
2. Embed bằng `intfloat/multilingual-e5-base`
3. Upsert lên Qdrant collection với full payload

**Payload gồm:** product_id, title, description, category, brand, price, image_url, tags, search_document

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/DATN/data'
print('Data directory:', DATA_DIR)
!ls -la "$DATA_DIR" | head -10

## 2. Cài đặt dependencies

In [ ]:
# Cài torch trước
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "torch", "torchvision", "torchaudio",
    "--index-url", "https://download.pytorch.org/whl/cu121"
])
print('>>> VUI LÒNG CHỌN Runtime > Restart runtime <<<')
import torch
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
# Cài các gói còn lại
%pip install -q -U \
    sentence-transformers \
    qdrant-client \
    pandas \
    tqdm

import torch
print('GPU:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

## 3. Cấu hình Qdrant

In [ ]:
import os, getpass

# Qdrant configuration
QDRANT_URL = 'http://qdrant.datn-nextgen-suggest.site'
QDRANT_API_KEY = getpass.getpass('QDRANT_API_KEY: ')

os.environ['QDRANT_API_KEY'] = QDRANT_API_KEY

print('Qdrant URL:', QDRANT_URL)

## 4. Load corpus từ CSV

In [ ]:
import pandas as pd
from pathlib import Path
import ast

CORPUS_CSV = Path('/content/drive/MyDrive/DATN/data/Dataset_DATN_28k.csv')
df = pd.read_csv(CORPUS_CSV)
print(f'Loaded {len(df)} products')
print('Columns:', list(df.columns))
df.head(2)

## 5. Build corpus text & payload

In [ ]:
def safe_parse_tags(val):
    """Parse tags column - có thể là list hoặc string."""
    if pd.isna(val):
        return []
    if isinstance(val, list):
        return val
    try:
        return ast.literal_eval(val)
    except:
        return [val] if val else []

def build_corpus_text(row):
    """Build text representation cho embedding."""
    parts = []
    if pd.notna(row.get('product_name')):
        parts.append(str(row['product_name']))
    if pd.notna(row.get('description')):
        parts.append(str(row['description']))
    if pd.notna(row.get('category_name')):
        parts.append(str(row['category_name']))
    if pd.notna(row.get('brand')):
        parts.append(str(row['brand']))
    return ' '.join(parts)

def build_payload(row, idx):
    """Build full payload cho Qdrant."""
    return {
        'product_id': str(row['product_id']),
        'title': str(row['product_name']) if pd.notna(row.get('product_name')) else '',
        'description': str(row['description']) if pd.notna(row.get('description')) else '',
        'category': str(row['category_name']) if pd.notna(row.get('category_name')) else '',
        'brand': str(row['brand']) if pd.notna(row.get('brand')) else '',
        'price': float(row['price']) if pd.notna(row.get('price')) else 0,
        'image_url': str(row['thumbnail_url']) if pd.notna(row.get('thumbnail_url')) else '',
        'tags': safe_parse_tags(row.get('tags')),
        'search_document': str(row['searchable_text']) if pd.notna(row.get('searchable_text')) else ''
    }

# Build corpus
corpus_ids = df['product_id'].tolist()
corpus_texts = [build_corpus_text(row) for _, row in df.iterrows()]
payloads = [build_payload(row, i) for i, (_, row) in enumerate(df.iterrows())]

print(f'Built {len(corpus_texts)} corpus items')
print('\nSample payload:')
for k, v in payloads[0].items():
    val_str = str(v)
    print(f'  {k}: {val_str[:100]}...' if len(val_str) > 100 else f'  {k}: {val_str}')

## 6. Embed corpus với E5-Base

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print('Loading model...')
model = SentenceTransformer('intfloat/multilingual-e5-base')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Model loaded. Using device: {device}')

# E5 yêu cầu prefix 'passage: '
PREFIX = 'passage: '
corpus_with_prefix = [PREFIX + text for text in corpus_texts]

# Encode in batches
BATCH_SIZE = 128
print(f'Embedding {len(corpus_with_prefix)} texts in batches of {BATCH_SIZE}...')

embeddings = model.encode(
    corpus_with_prefix,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_tensor=True,
    device=device
)

print(f'Embeddings shape: {embeddings.shape}')

## 7. Upload to Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import numpy as np

# Connect to Qdrant
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=120
)

COLLECTION_NAME = 'products_corpus_v2'
EMBED_DIM = 768  # E5-base output dimension

# Check if collection exists
collections = client.get_collections().collections
collection_names = [c.name for c in collections]
print(f'Existing collections: {collection_names}')

# Delete if exists (recreate with new data)
if COLLECTION_NAME in collection_names:
    print(f'Deleting existing collection: {COLLECTION_NAME}')
    client.delete_collection(COLLECTION_NAME)

# Create collection
print(f'Creating collection: {COLLECTION_NAME}')
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=EMBED_DIM,
        distance=Distance.COSINE
    )
)
print('Collection created!')

In [ ]:
# Upsert embeddings với full payload
from tqdm import tqdm

BATCH_SIZE = 100  # Qdrant recommended batch size
points_batch = []

print(f'Upserting {len(corpus_ids)} vectors to Qdrant...')

for i, (pid, emb, payload) in enumerate(tqdm(zip(corpus_ids, embeddings, payloads), total=len(corpus_ids))):
    point = PointStruct(
        id=str(pid),  # Use product_id as point ID
        vector=emb.cpu().numpy().tolist(),
        payload=payload
    )
    points_batch.append(point)
    
    # Upsert when batch is full
    if len(points_batch) >= BATCH_SIZE:
        client.upsert(
            collection_name=COLLECTION_NAME,
            points=points_batch
        )
        points_batch = []

# Upsert remaining points
if points_batch:
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points_batch
    )

print(f'Uploaded {len(corpus_ids)} vectors to Qdrant collection: {COLLECTION_NAME}')

## 8. Verify upload

In [ ]:
# Check collection info
info = client.get_collection(COLLECTION_NAME)
print(f'Collection: {COLLECTION_NAME}')
print(f'Vectors count: {info.vectors_count}')
print(f'Points count: {info.points_count}')

# Test search
test_query = "son shu uemura"
test_emb = model.encode(f'query: {test_query}', device=device)

results = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=test_emb.tolist(),
    limit=3
)

print(f'\nTest search for: "{test_query}"')
for r in results:
    print(f'\n  Score: {r.score:.4f}')
    print(f'  ID: {r.id}')
    print(f'  Title: {r.payload.get("title", "N/A")[:80]}...')
    print(f'  Category: {r.payload.get("category", "N/A")}')
    print(f'  Price: {r.payload.get("price", "N/A")}')

## Summary

Đã upload **28,212 sản phẩm** lên Qdrant:
- Collection: `products_corpus_v2`
- Vector dimension: 768 (E5-base)
- Distance: COSINE

**Payload fields:**
- product_id, title, description, category, brand
- price, image_url, tags, search_document